# 2. Erarbeitungsphase (Umsetzung) – A³-Zyklus im QUA³CK-Prozess

In dieser Phase wird das Projekt praktisch umgesetzt. Der Fokus liegt auf dem iterativen **A³-Zyklus** (Algorithm Selection → Adapting Features → Adjusting Hyperparameters). Ziel ist es, verschiedene Modelle systematisch zu trainieren, vergleichbar zu evaluieren und schrittweise zu verbessern. Der A³-Zyklus ist damit der Kern des maschinellen Lernprozesses: Auf Basis der Erkenntnisse aus der EDA (U-Phase) werden geeignete Algorithmen ausgewählt, die Daten in eine lernbare Form überführt und anschließend Hyperparameter so angepasst, dass die Performance auf validierten Daten maximiert wird. Die Iterationen werden nachvollziehbar dokumentiert, um Reproduzierbarkeit und Vergleichbarkeit sicherzustellen.

Die Umsetzung orientiert sich an den zuvor definierten Zielen: (1) robuste Vorhersageleistung, (2) leakage-freies Vorgehen über Pipelines, (3) transparente Modellvergleiche mit geeigneten Metriken (insbesondere F1/PR-AUC bei möglicher Klassenungleichheit) sowie (4) Berücksichtigung von Interpretierbarkeit und Rechenaufwand. Als Ressourcen werden ein lokales Python-Setup (Jupyter Notebook), scikit-learn-Pipelines sowie Standard-Hardware genutzt. Potenzielle Risiken wie Klassenimbalance, Datenqualität (z. B. `TotalCharges` als String), Overfitting durch zu komplexe Modelle und Datenleckage werden aktiv überwacht. Der Projektfortschritt wird über klare Zwischenergebnisse (Baseline → Pipeline → Modellvergleich → Tuning → Best Model) kontrolliert.

---

## 2.1 Reihenfolge der Umsetzung 

1. **Baseline & Splits festlegen**  
   - Train/Test-Split (stratifiziert) + optional CV-Strategie definieren  
   - Baseline mit `DummyClassifier` erstellen (Referenz)

2. **A¹ – Algorithm Selection (Modellauswahl)**  
   - Kandidatenmodelle definieren (z. B. Logistic Regression, Random Forest, Gradient Boosting)  
   - Einheitliche Evaluation (gleiche Splits, gleiche Metriken)

3. **A² – Adapting Features (Feature-Anpassung & Preprocessing)**  
   - `ColumnTransformer` + Pipeline: Encoding (OneHot), Imputation, Skalierung  
   - Optional: Feature Engineering (z. B. neue Variablen, sauberes Casting von `TotalCharges`)  
   - Leakage vermeiden: alle Transformationen **nur innerhalb der Pipeline**

4. **A³ – Adjusting Hyperparameters (Tuning)**  
   - `GridSearchCV`/`RandomizedSearchCV` je Modell  
   - Vergleich Default vs. Tuned  
   - Ergebnis-Logging (Parameter, Scores, Laufzeit)

5. **Zwischenfazit & Entscheidung**  
   - Bestes Modell nach primärer Metrik auswählen (z. B. F1/PR-AUC)  
   - Interpretierbarkeit & Rechenaufwand als Sekundärkriterien dokumentieren

---

## 2.2 Realisierung 

Die Daten werden mit **pandas** aus der CSV eingelesen und im Anschluss in ein klares Feature/Target-Schema überführt (`X`, `y`). Danach wird eine **reproduzierbare Validierungsstrategie** (stratifizierter Split und/oder Stratified K-Fold) festgelegt. Der Modellierungsprozess erfolgt streng **leakage-frei** über scikit-learn-Pipelines: Kategoriale Features werden per One-Hot-Encoding transformiert, numerische Features imputiert und – falls erforderlich – skaliert. Auf dieser Basis werden mehrere Kandidatenmodelle trainiert und anhand standardisierter Metriken (ROC-AUC, F1, Precision/Recall, PR-AUC) verglichen. 

Im iterativen A³-Zyklus werden zunächst Default-Modelle als Ausgangspunkt getestet (Algorithm Selection). Danach werden Features und Preprocessing systematisch angepasst (Adapting Features), z. B. Umgang mit fehlenden Werten, korrekte Datentypisierung von `TotalCharges` und optionales Threshold-Tuning. Anschließend werden relevante Hyperparameter mit Cross-Validation optimiert (Adjusting Hyperparameters), um die Generalisierung zu verbessern und Overfitting zu vermeiden. Alle Experimente werden dokumentiert (Modell, Parameter, Scores, Laufzeiten), sodass Entscheidungen nachvollziehbar bleiben und die Ergebnisse reproduzierbar sind.

---

## 2.3 Ressourcen, Statusüberwachung und Fortschritt

**Ressourcen**
- Python (Jupyter Notebook), `pandas`, `scikit-learn`, `matplotlib/seaborn`
- Lokale Rechenressourcen (CPU), keine GPU erforderlich

**Status & Fortschritt (Meilensteine)**
- M1: Daten geladen + Target/Features definiert  
- M2: Baseline (DummyClassifier) + erste Metriken  
- M3: Pipeline steht (Preprocessing leakage-frei)  
- M4: Modellvergleich (LogReg vs. RF vs. GB) abgeschlossen  
- M5: Hyperparameter-Tuning + bestes Modell bestimmt  
- M6: Zwischenstand dokumentiert (Portfolioteil 2)

---

## 2.4 Potenzielle Probleme, Risiken und Verbesserungsmöglichkeiten

- **Klassenimbalance:** Accuracy kann irreführend sein → Fokus auf F1/PR-AUC, stratifizierte Splits, optional `class_weight='balanced'` oder Threshold-Tuning.  
- **Datenqualität:** `TotalCharges` kann als String vorliegen → Casting + Umgang mit fehlenden/leerem String.  
- **Overfitting:** zu komplexe Modelle/zu viele Parameter → Cross-Validation, Regularisierung, begrenzte Baumtiefe, Early Stopping (falls genutzt).  
- **Datenleckage:** Preprocessing außerhalb der Pipeline → strikt Pipeline/ColumnTransformer nutzen.  
- **Interpretierbarkeit:** Ensemble-Modelle schwerer erklärbar → Permutation Importance, ggf. SHAP optional.

---

## 2.5 Digitaler Entwurf / Zwischenschritt (für die Abgabe)

Als digitaler Zwischenschritt wird ein reproduzierbarer Modell-Prototyp umgesetzt, bestehend aus:
- **Preprocessing-Pipeline** (ColumnTransformer: Imputation + OneHot + Scaling)  
- **Baseline-Modell** (DummyClassifier)  
- **Erster Modellvergleich** (Logistic Regression, Random Forest, Gradient Boosting) mit einheitlichen Metriken  

Die folgenden Code-Skizzen dienen als dokumentierter Zwischenstand (A³-Startpunkt) und werden im Notebook ausgeführt.

---

## 2.6 Code-Skizze: Baseline, Pipeline und erster Modellvergleich (Zwischenstand)

> Hinweis: Dies ist ein prototypischer Zwischenschritt. Spaltennamen müssen zum Datensatz passen.

```python
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import make_scorer, f1_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1) Daten laden
data = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2) Target und Features
y = (data["Churn"] == "Yes").astype(int)
X = data.drop(columns=["Churn"])

# 3) Spaltentypen (vereinfachte Heuristik)
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# 4) Preprocessing (leakage-frei)
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

# 5) Modelle (Algorithm Selection)
models = {
    "Dummy (MostFrequent)": DummyClassifier(strategy="most_frequent"),
    "LogReg": LogisticRegression(max_iter=200, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "GradBoost": GradientBoostingClassifier(random_state=42)
}

# 6) Evaluation Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "f1": make_scorer(f1_score),
    "precision": "precision",
    "recall": "recall"
}

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean()
    })

pd.DataFrame(results).sort_values(by="f1_mean", ascending=False)